# Text Feature Engineering Assignment

## Problem Statement
You are building a Text Processing Pipeline for a company that wants to analyze real user-generated
text data such as reviews and comments, and convert them into numerical features for machine
learning models. Your task is to collect real-world data and implement One Hot Encoding, Bag of
Words, and TF-IDF.

## Dataset Collection (Real world)
Dataset Collection (Real-world)
1. Scrape product reviews from Flipkart or Amazon (minimum 100 reviews)
2. You can use Selenium, BeautifulSoup, or any reliable scraping method
3. Store data in CSV format with at least one column: review_text
4. Ensure data is clean and readable
5. Do not scrape restricted or sensitive content

---
### Kaggle dataset 7817_1.csv
I initially tried to use GitHub Copilot to create a webscraper using BeautifulSoup and Selenium.  When trying to run the scraper, it kept coming into contact with bot protection.  I wanted to focus my own understanding more on the content from the previous classes so instead downloaded this Kaggle dataset that contained reviews.
https://www.kaggle.com/code/jalesummak/amazon-reviews-topic-modeling-with-nlp-nmf-lda

###  Webscraped data
I finally got a webscraper code functioning with Selenium.  I scraped real customer review data of the GAOMON S620 OSU Signature Graphics Tablet.  It builts a structured dataset from raw HTML content.  One can use either dataset to run the code.


## Task 1: Preprocessing

In [17]:
import pandas as pd

df_raw = pd.read_csv('../data/7817_1.csv') # Use "../data/7817_1.csv" for the kaggle dataset, and "../data/amazon_reviews.csv" for the amazon reviews dataset.
df = df_raw.copy()

review_text_column = 'reviews.text' # Use 'reviews.text' for the kaggle dataset, and 'review_text' for the amazon reviews dataset.
review_text_rating = 'reviews.rating' # Use 'reviews.rating' for the kaggle dataset, and 'review_rating' for the amazon reviews dataset.

reviews = df[review_text_column].tolist() # Use 'reviews.text' for the kaggle dataset, and 'review_text' for the amazon reviews dataset.

### Manual Preprocessing (from classes)

In [2]:
# Get all words from each review, convert to lower case, remove punctuation, and tokenise
import string

translator = str.maketrans('', '', string.punctuation)
df["tokens"] = [review.lower().translate(translator).split() for review in reviews]
df["tokens"]

0      [wow, im, pleasantly, surprised, i, previously...
1      [very, impressed, with, this, signature, table...
2      [i, own, a, macbook, pro, 14”, with, an, m5, c...
3      [i, love, this, tablet, the, pen, feels, solid...
4      [my, wife, needed, a, tablet, to, complete, a,...
                             ...                        
495    [affordable, tablet, that, just, works, my, ex...
496    [i, bought, this, tablet, a, few, years, back,...
497    [pretty, efficient, very, effective, it, is, a...
498    [its, good, for, drawing, the, buttons, are, u...
499    [great, product, it, works, straifht, out, of,...
Name: tokens, Length: 500, dtype: object

In [3]:
# Removing stopwords from tokens
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
tokens = df["tokens"].tolist()
filtered_tokens = [
    [word for word in sentence if word not in ENGLISH_STOP_WORDS]
    for sentence in tokens
]
filtered_tokens[0]

['wow',
 'im',
 'pleasantly',
 'surprised',
 'previously',
 'owned',
 'expensive',
 'wacom',
 'tablet',
 'years',
 'frustrations',
 'relation',
 'drivers',
 'glitchy',
 'behaviour',
 'finally',
 'decided',
 'chance',
 'different',
 'brand',
 'say',
 'im',
 'blown',
 'away',
 'price',
 'point',
 'excellent',
 'dont',
 'know',
 'anybody',
 'need',
 'spend',
 'money',
 'does',
 'want',
 'does',
 'software',
 'great',
 'nice',
 'simple',
 'works',
 'clutter',
 'happy',
 'purchase']

In [4]:
# Lemmatization of tokens
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
df["lemmatized_tokens"] = [
    [lemmatizer.lemmatize(word) for word in sentence]
    for sentence in filtered_tokens
]
df["lemmatized_tokens"][0]

['wow',
 'im',
 'pleasantly',
 'surprised',
 'previously',
 'owned',
 'expensive',
 'wacom',
 'tablet',
 'year',
 'frustration',
 'relation',
 'driver',
 'glitchy',
 'behaviour',
 'finally',
 'decided',
 'chance',
 'different',
 'brand',
 'say',
 'im',
 'blown',
 'away',
 'price',
 'point',
 'excellent',
 'dont',
 'know',
 'anybody',
 'need',
 'spend',
 'money',
 'doe',
 'want',
 'doe',
 'software',
 'great',
 'nice',
 'simple',
 'work',
 'clutter',
 'happy',
 'purchase']

In [5]:
# Get all words from each review
lemmatized_tokens = df["lemmatized_tokens"].tolist()
all_words = [[word] for sentence in lemmatized_tokens for word in sentence]
all_words

[['wow'],
 ['im'],
 ['pleasantly'],
 ['surprised'],
 ['previously'],
 ['owned'],
 ['expensive'],
 ['wacom'],
 ['tablet'],
 ['year'],
 ['frustration'],
 ['relation'],
 ['driver'],
 ['glitchy'],
 ['behaviour'],
 ['finally'],
 ['decided'],
 ['chance'],
 ['different'],
 ['brand'],
 ['say'],
 ['im'],
 ['blown'],
 ['away'],
 ['price'],
 ['point'],
 ['excellent'],
 ['dont'],
 ['know'],
 ['anybody'],
 ['need'],
 ['spend'],
 ['money'],
 ['doe'],
 ['want'],
 ['doe'],
 ['software'],
 ['great'],
 ['nice'],
 ['simple'],
 ['work'],
 ['clutter'],
 ['happy'],
 ['purchase'],
 ['impressed'],
 ['signature'],
 ['tablet'],
 ['compatible'],
 ['window'],
 ['11'],
 ['install'],
 ['just'],
 ['download'],
 ['driver'],
 ['website'],
 ['tip'],
 ['run'],
 ['administrator'],
 ['installing'],
 ['driver'],
 ['user'],
 ['use'],
 ['pc'],
 ['tablet'],
 ['light'],
 ['weight'],
 ['decent'],
 ['practical'],
 ['sized'],
 ['stylus'],
 ['lovely'],
 ['grip'],
 ['responsive'],
 ['tablet'],
 ['easy'],
 ['use'],
 ['setup'],
 ['ma

### Use NLTK library for preprocessing instead (better for pipelines)

In [15]:
import re
import nltk

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [18]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#lowercase
df["clean_text"] = df[review_text_column].str.lower()

#remove punctuation
df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r'[^\w\s]', '', x))

#tokenization
df["tokens"] = df["clean_text"].apply(word_tokenize)

#remove stopwords
stop_words = set(stopwords.words('english'))
df["tokens"] = df["tokens"].apply(
    lambda x: [word for word in x if word not in stop_words])  

#lemmatization
lemmatizer = WordNetLemmatizer()
df["lemmatized_tokens"] = df["tokens"].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x])

# Final output
df["final_text"] = df["lemmatized_tokens"].apply(lambda x: ' '.join(x))

#Review the final output
print(f"Final cleaned text:\n{df.head()}")

Final cleaned text:
                     id       asins   brand                  categories  \
0  AVpe7AsMilAPnD_xQ78G  B00QJDU3KY  Amazon  Amazon Devices,mazon.co.uk   
1  AVpe7AsMilAPnD_xQ78G  B00QJDU3KY  Amazon  Amazon Devices,mazon.co.uk   
2  AVpe7AsMilAPnD_xQ78G  B00QJDU3KY  Amazon  Amazon Devices,mazon.co.uk   
3  AVpe7AsMilAPnD_xQ78G  B00QJDU3KY  Amazon  Amazon Devices,mazon.co.uk   
4  AVpe7AsMilAPnD_xQ78G  B00QJDU3KY  Amazon  Amazon Devices,mazon.co.uk   

  colors             dateAdded           dateUpdated  \
0    NaN  2016-03-08T20:21:53Z  2017-07-18T23:52:58Z   
1    NaN  2016-03-08T20:21:53Z  2017-07-18T23:52:58Z   
2    NaN  2016-03-08T20:21:53Z  2017-07-18T23:52:58Z   
3    NaN  2016-03-08T20:21:53Z  2017-07-18T23:52:58Z   
4    NaN  2016-03-08T20:21:53Z  2017-07-18T23:52:58Z   

                  dimension  ean                         keys  ...  \
0  169 mm x 117 mm x 9.1 mm  NaN  kindlepaperwhite/b00qjdu3ky  ...   
1  169 mm x 117 mm x 9.1 mm  NaN  kindlepaperwhite/b

## Task 2:  Vocabulary Creation

### Using Scikit-learn CountVectorizor

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df["final_text"])

# sum up counts of each word across all documents
word_counts = np.asarray(X.sum(axis=0)).flatten()

vocabulary = vectorizer.get_feature_names_out()

# pair words with counts
word_freq = list(zip(vocabulary, word_counts))

# sort by frequency
word_freq_sorted = sorted(
    [(w, int(c)) for w, c in zip(
        vocabulary,
        word_counts
    )],
    key=lambda x: x[1],
    reverse=True
)

# top 10 most common words

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Most common words: {word_freq_sorted[:20]}")


Vocabulary size: 7244
Most common words: [('kindle', 1487), ('amazon', 1403), ('fire', 1376), ('like', 1244), ('use', 945), ('device', 906), ('one', 866), ('tablet', 837), ('sound', 833), ('tv', 809), ('headphone', 808), ('great', 798), ('im', 776), ('dont', 749), ('prime', 736), ('review', 736), ('year', 723), ('would', 705), ('screen', 630), ('apple', 620)]


### Using collections Counter

In [20]:
from collections import Counter
all_words = " ".join(df["final_text"]).split()
vocabulary = Counter(all_words)

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Most common words: {vocabulary.most_common(20)}")

Vocabulary size: 7267
Most common words: [('kindle', 1487), ('amazon', 1403), ('fire', 1376), ('like', 1244), ('use', 945), ('device', 906), ('one', 866), ('tablet', 837), ('sound', 833), ('tv', 809), ('headphone', 808), ('great', 798), ('im', 776), ('dont', 749), ('review', 736), ('prime', 736), ('year', 723), ('would', 705), ('screen', 630), ('apple', 620)]


## Task 3: Feature Engineering


In [21]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Build document strings from cleaned + lemmatized tokens
documents = [" ".join(sentence) for sentence in df["lemmatized_tokens"] if sentence]
token_documents = [sentence for sentence in df["lemmatized_tokens"] if sentence]

if not documents:
    raise ValueError("No documents found. Run preprocessing cells before Task 3.")

# Use Task 2 vocabulary if available, otherwise derive from tokenized documents
if "vocabulary" not in globals() or len(vocabulary) == 0:
    vocabulary = sorted({word for sentence in token_documents for word in sentence})

# 1) One Hot Encoding (document-level vector)
vocab_index = {word: idx for idx, word in enumerate(vocabulary)}
ohe_matrix = np.zeros((len(token_documents), len(vocabulary)), dtype=int)

for doc_idx, sentence in enumerate(token_documents):
    for word in set(sentence):  # set() ensures binary presence per document
        col_idx = vocab_index.get(word)
        if col_idx is not None:
            ohe_matrix[doc_idx, col_idx] = 1

ohe_df = pd.DataFrame(ohe_matrix, columns=vocabulary)
print("One-Hot shape:", ohe_df.shape)
display(ohe_df.iloc[:5, :20])

# 2) Bag of Words (CountVectorizer)
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(documents)
bow_features = count_vectorizer.get_feature_names_out()
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_features)
print("BoW shape:", bow_df.shape)
display(bow_df.iloc[:5, :20])

# 3) TF-IDF (TfidfVectorizer)
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
tfidf_features = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)
print("TF-IDF shape:", tfidf_df.shape)
display(tfidf_df.iloc[:5, :20])

# Quick sanity check: show non-zero features for first document
sample_doc_index = 0
ohe_terms = ohe_df.columns[ohe_df.iloc[sample_doc_index].values == 1].tolist()[:50]
bow_nonzero = bow_df.iloc[sample_doc_index][bow_df.iloc[sample_doc_index] > 0].sort_values(ascending=False).head(10)
tfidf_nonzero = tfidf_df.iloc[sample_doc_index][tfidf_df.iloc[sample_doc_index] > 0].sort_values(ascending=False).head(10)

print("\nSample document index:", sample_doc_index)
print("OHE non-zero terms (first 50):", ohe_terms)
print("\nBoW top terms for sample doc:")
display(bow_nonzero.to_frame("count"))
print("TF-IDF top terms for sample doc:")
display(tfidf_nonzero.to_frame("tfidf"))

One-Hot shape: (1596, 7267)


,initially,trouble,deciding,paperwhite,voyage,review,less,said,thing,great,spending,money,go,voyagefortunately,friend,owned,ended,buying,basis,model
0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0
2,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0


BoW shape: (1596, 7244)


,029,034,035,03mp,04,040714after,045,07082017,07112017,09gb,10,100,1000,10000,100000,1000ma,100mbps,101,1011,1015
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


TF-IDF shape: (1596, 7244)


,029,034,035,03mp,04,040714after,045,07082017,07112017,09gb,10,100,1000,10000,100000,1000ma,100mbps,101,1011,1015
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Sample document index: 0
OHE non-zero terms (first 50): ['initially', 'trouble', 'deciding', 'paperwhite', 'voyage', 'review', 'less', 'said', 'thing', 'great', 'spending', 'money', 'go', 'voyagefortunately', 'friend', 'owned', 'ended', 'buying', 'basis', 'model', '300', 'ppi', '80', 'dollar', 'jump', 'turn', 'pricey', 'page', 'press', 'isnt', 'always', 'sensitive', 'fine', 'specific', 'setting', 'dont', 'need', 'auto', 'light', 'adjustmentits', 'week', 'loving', 'regret', 'touch', 'screen', 'receptive', 'easy', 'use', 'keep', 'regardless']

BoW top terms for sample doc:


,count
paperwhite,4
setting,3
light,3
time,3
day,2
need,2
specific,2
shipping,2
voyage,2
300,1


TF-IDF top terms for sample doc:


,tfidf
paperwhite,0.302753
setting,0.282611
light,0.216457
shipping,0.204570
voyage,0.204570
specific,0.158924
time,0.146933
adjustmentits,0.139618
voyagefortunately,0.139618
receptive,0.139618


## Task 4: Comparison Analysis

Create a comparison table for OHE, BoW, and TF-IDF, then identify the most important TF-IDF words. Explain why common words receive lower TF-IDF weight.

In [22]:
# Task 4: Comparison table + TF-IDF importance analysis
import pandas as pd

# Build a concise method comparison table
comparison_df = pd.DataFrame(
    [
        {
            "Method": "One Hot Encoding (OHE)",
            "Vector Values": "Binary (0 or 1)",
            "Uses Frequency?": "No",
            "Considers Importance Across Docs?": "No",
            "Sparsity": "Very High",
            "Strength": "Simple and interpretable; captures presence",
            "Limitation": "Ignores term frequency and global importance",
        },
        {
            "Method": "Bag of Words (BoW)",
            "Vector Values": "Integer counts",
            "Uses Frequency?": "Yes (within document)",
            "Considers Importance Across Docs?": "No",
            "Sparsity": "High",
            "Strength": "Captures term frequency",
            "Limitation": "Common words can dominate",
        },
        {
            "Method": "TF-IDF",
            "Vector Values": "Weighted real values",
            "Uses Frequency?": "Yes (TF)",
            "Considers Importance Across Docs?": "Yes (IDF)",
            "Sparsity": "High",
            "Strength": "Highlights informative, discriminative words",
            "Limitation": "Less intuitive than simple counts",
        },
    ]
)

display(comparison_df)

# Find globally most important TF-IDF words by average TF-IDF score
avg_tfidf = tfidf_df.mean(axis=0).sort_values(ascending=False)
top_tfidf_words = avg_tfidf.head(15).to_frame("avg_tfidf_score")
print("Top 15 most important words by average TF-IDF across corpus:")
display(top_tfidf_words)

# Show why common words get lower weight by checking document frequency + IDF formula intuition
doc_presence = (bow_df > 0).sum(axis=0).sort_values(ascending=False)
common_words = doc_presence.head(10).to_frame("document_count")
common_words["document_ratio"] = common_words["document_count"] / len(documents)

print("Most common words by document frequency:")
display(common_words)

print("Explanation:")
print(
    "TF-IDF gives high scores to words that appear frequently in a specific document (high TF) "
    "but not in too many documents overall (high IDF)."
 )
print(
    "Words that occur in many documents have low IDF, so their final TF-IDF weight is reduced. "
    "That is why very common words receive lower importance."
 )

,Method,Vector Values,Uses Frequency?,Considers Importance Across Docs?,Sparsity,Strength,Limitation
0,One Hot Encoding (OHE),Binary (0 or 1),No,No,Very High,Simple and interpretable; captures presence,Ignores term frequency and global importance
1,Bag of Words (BoW),Integer counts,Yes (within document),No,High,Captures term frequency,Common words can dominate
2,TF-IDF,Weighted real values,Yes (TF),Yes (IDF),High,"Highlights informative, discriminative words",Less intuitive than simple counts


Top 15 most important words by average TF-IDF across corpus:


,avg_tfidf_score
kindle,0.037305
great,0.036198
fire,0.033180
sound,0.032361
amazon,0.031381
like,0.030311
use,0.028829
love,0.027855
headphone,0.027541
echo,0.026725


Most common words by document frequency:


,document_count,document_ratio
amazon,685,0.429198
like,599,0.375313
great,558,0.349624
use,523,0.327694
one,478,0.299499
would,436,0.273183
sound,436,0.273183
work,428,0.268170
device,422,0.264411
fire,421,0.263784


Explanation:
TF-IDF gives high scores to words that appear frequently in a specific document (high TF) but not in too many documents overall (high IDF).
Words that occur in many documents have low IDF, so their final TF-IDF weight is reduced. That is why very common words receive lower importance.


Why common words receive lower TF-IDF weight:

TF-IDF combines term frequency and inverse document frequency:

$TF-IDF(t,d)=TF(t,d)×IDF(t)$

$IDF(t)≈log⁡(Ndf(t))$

If a word appears in many documents, df(t)df(t) is high, so IDF is low.
That lowers the final TF-IDF score, even when the word appears often.
So TF-IDF emphasizes discriminative words, not globally common words.


## Task 5: Sparse Matrix Analysis

In [23]:

import numpy as np
import pandas as pd

def sparsity_percent(matrix_like) -> float:
    arr = matrix_like.to_numpy() if hasattr(matrix_like, "to_numpy") else np.asarray(matrix_like)
    total_elements = arr.size
    zero_elements = np.count_nonzero(arr == 0)
    return (zero_elements / total_elements) * 100

analysis_df = pd.DataFrame(
    [
        {
            "Matrix": "One Hot Encoding (OHE)",
            "Shape": f"{ohe_df.shape[0]} x {ohe_df.shape[1]}",
            "Sparsity (%)": round(sparsity_percent(ohe_df), 2),
        },
        {
            "Matrix": "Bag of Words (BoW)",
            "Shape": f"{bow_df.shape[0]} x {bow_df.shape[1]}",
            "Sparsity (%)": round(sparsity_percent(bow_df), 2),
        },
        {
            "Matrix": "TF-IDF",
            "Shape": f"{tfidf_df.shape[0]} x {tfidf_df.shape[1]}",
            "Sparsity (%)": round(sparsity_percent(tfidf_df), 2),
        },
    ]
)

print("Sparse Matrix Summary:")
display(analysis_df)

print("\nWhy sparse matrices can be inefficient at large scale:")
print(
    "1. Most entries are zeros, so dense storage wastes memory and increases storage costs."
 )
print(
    "2. Dense computations process many zero values, causing unnecessary CPU/GPU work and slower training/inference."
 )
print(
    "3. Moving large sparse vectors across memory/network increases bandwidth pressure in distributed systems."
 )
print(
    "4. As vocabulary and dataset size grow, dimensionality explodes, making scaling difficult without sparse formats, feature selection, or embeddings."
 )

Sparse Matrix Summary:


,Matrix,Shape,Sparsity (%)
0,One Hot Encoding (OHE),1596 x 7267,99.12
1,Bag of Words (BoW),1596 x 7244,99.13
2,TF-IDF,1596 x 7244,99.13



Why sparse matrices can be inefficient at large scale:
1. Most entries are zeros, so dense storage wastes memory and increases storage costs.
2. Dense computations process many zero values, causing unnecessary CPU/GPU work and slower training/inference.
3. Moving large sparse vectors across memory/network increases bandwidth pressure in distributed systems.
4. As vocabulary and dataset size grow, dimensionality explodes, making scaling difficult without sparse formats, feature selection, or embeddings.


## Task 6: Real-world Questions

In [24]:
# Task 6 demo: BoW similarity for semantically similar sentences
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

semantic_sentences = [
    "The movie was good and enjoyable.",
    "The film was excellent and fun.",
    "The weather is sunny today.",
]

bow_demo = CountVectorizer(stop_words="english")
X_bow_demo = bow_demo.fit_transform(semantic_sentences)
bow_sim = cosine_similarity(X_bow_demo)

tfidf_demo = TfidfVectorizer(stop_words="english")
X_tfidf_demo = tfidf_demo.fit_transform(semantic_sentences)
tfidf_sim = cosine_similarity(X_tfidf_demo)

labels = ["A: movie good", "B: film excellent", "C: weather sunny"]
bow_sim_df = pd.DataFrame(bow_sim, index=labels, columns=labels)
tfidf_sim_df = pd.DataFrame(tfidf_sim, index=labels, columns=labels)

print("BoW cosine similarity matrix:")
display(bow_sim_df)
print("TF-IDF cosine similarity matrix:")
display(tfidf_sim_df)

print("Observation:")
print(
    "A and B are semantically similar, but lexical overlap is low, so BoW/TF-IDF similarity remains limited. "
    "This illustrates why these methods struggle with synonym-level meaning."
 )

BoW cosine similarity matrix:


,A: movie good,B: film excellent,C: weather sunny
A: movie good,1.0,0.0,0.0
B: film excellent,0.0,1.0,0.0
C: weather sunny,0.0,0.0,1.0


TF-IDF cosine similarity matrix:


,A: movie good,B: film excellent,C: weather sunny
A: movie good,1.0,0.0,0.0
B: film excellent,0.0,1.0,0.0
C: weather sunny,0.0,0.0,1.0


Observation:
A and B are semantically similar, but lexical overlap is low, so BoW/TF-IDF similarity remains limited. This illustrates why these methods struggle with synonym-level meaning.


## Task 7: Mini Use Case

In [26]:
# Task 7: Sentiment classification mini use case
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Prepare labels from ratings
work_df = df_raw[[review_text_column, review_text_rating]].copy()
work_df[review_text_column] = work_df[review_text_column].fillna("").astype(str)
work_df[review_text_rating] = pd.to_numeric(work_df[review_text_rating], errors="coerce")
work_df = work_df.dropna(subset=[review_text_rating])

# Keep only clear positive/negative classes
work_df = work_df[work_df[review_text_rating].isin([1, 2, 4, 5])].copy()
work_df["sentiment"] = (work_df[review_text_rating] >= 4).astype(int)  # 1=positive, 0=negative

X = work_df[review_text_column]
y = work_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class-balanced logistic regression handles the strong class imbalance better.
bow_lr = Pipeline(
    [
        ("vectorizer", CountVectorizer(stop_words="english", min_df=2)),
        (
            "model",
            LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced"),
        ),
    ]
)

tfidf_lr = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(stop_words="english", min_df=2)),
        (
            "model",
            LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced"),
        ),
    ]
)

bow_lr.fit(X_train, y_train)
tfidf_lr.fit(X_train, y_train)

bow_pred = bow_lr.predict(X_test)
tfidf_pred = tfidf_lr.predict(X_test)

results_df = pd.DataFrame(
    [
        {
            "Feature": "BoW + Logistic Regression",
            "Accuracy": round(accuracy_score(y_test, bow_pred), 4),
            "F1 (Positive)": round(f1_score(y_test, bow_pred, pos_label=1), 4),
            "F1 (Macro)": round(f1_score(y_test, bow_pred, average="macro"), 4),
        },
        {
            "Feature": "TF-IDF + Logistic Regression",
            "Accuracy": round(accuracy_score(y_test, tfidf_pred), 4),
            "F1 (Positive)": round(f1_score(y_test, tfidf_pred, pos_label=1), 4),
            "F1 (Macro)": round(f1_score(y_test, tfidf_pred, average="macro"), 4),
        },
    ]
).sort_values(by=["F1 (Macro)", "F1 (Positive)", "Accuracy"], ascending=False)

print("Dataset size used for Task 7:", len(work_df))
print("Class distribution (0=negative, 1=positive):")
display(y.value_counts().sort_index().rename("count").to_frame())

print("Model performance comparison:")
display(results_df)

print("Detailed report - BoW + Logistic Regression")
print(classification_report(y_test, bow_pred, target_names=["negative", "positive"], zero_division=0))

print("Detailed report - TF-IDF + Logistic Regression")
print(classification_report(y_test, tfidf_pred, target_names=["negative", "positive"], zero_division=0))

Dataset size used for Task 7: 1053
Class distribution (0=negative, 1=positive):


,count
sentiment,
0,76
1,977


Model performance comparison:


,Feature,Accuracy,F1 (Positive),F1 (Macro)
1,TF-IDF + Logistic Regression,0.9479,0.9719,0.8085
0,BoW + Logistic Regression,0.9336,0.9648,0.6907


Detailed report - BoW + Logistic Regression
              precision    recall  f1-score   support

    negative       0.56      0.33      0.42        15
    positive       0.95      0.98      0.96       196

    accuracy                           0.93       211
   macro avg       0.75      0.66      0.69       211
weighted avg       0.92      0.93      0.93       211

Detailed report - TF-IDF + Logistic Regression
              precision    recall  f1-score   support

    negative       0.62      0.67      0.65        15
    positive       0.97      0.97      0.97       196

    accuracy                           0.95       211
   macro avg       0.80      0.82      0.81       211
weighted avg       0.95      0.95      0.95       211



## Final Summary and Deliverables Checklist

### Best Model from Task 7
Based on the sentiment classification experiment, **TF-IDF + Logistic Regression** is the best-performing model on the Kaggle dataset.

Evidence from the latest run:
1. Accuracy: **0.9479** (higher than BoW: 0.9336)
2. F1 (Positive): **0.9719** (higher than BoW: 0.9648)
3. F1 (Macro): **0.8085** (higher than BoW: 0.6907)

Why this model performed best:
1. TF-IDF down-weights very common words and emphasizes discriminative terms.
2. Logistic Regression is strong for high-dimensional sparse text features.
3. Class-balanced training improved minority-class handling and made comparison fairer.

Using the Amazon review dataset that was scraped, I would suggest that there is no good model given the data.  Amazon bias products with review ratings of 4 and above.  This makes the data severly unbalanced. This is reflected in the implementation and metrics which are all perfect.  Looking at the reviews scraped, only 14% out of 22820 are 3 stars or below.  More work would be needed to obtain a more suitable dataset of reviews to adequately score sentiment.  The scraping itself is biased to the review page that is sorted by "Top reviews" which skews the dataset further.

### Deliverables Status
1. **Python notebook (.ipynb)**: Completed in this file.
2. **Clean and modular code**: Completed. Code is organized task-by-task with reusable preprocessing outputs and clear sectioning.
3. **Scraped dataset (CSV)**: CSV file included and used: `7817_1.csv`.
4. **Output screenshots**: Please capture screenshots of key outputs from Tasks 3 to 7 (feature matrices, comparison tables, sparse analysis, model metrics).
5. **Short report (1-2 pages) with observations and conclusions**: Completed in notebook narrative sections (Tasks 4 to 7 + this final summary). You can export these sections into a short PDF/Doc report.

### Final Observation
For this assignment dataset and setup, classical NLP with **TF-IDF + Logistic Regression** provides a strong, interpretable baseline and outperforms BoW across key evaluation metrics.